# PDFD

## 1. Data Preparation

Load the processed market dataset and recover the return series used to estimate drawdown-event probabilities. In Libélula, PDFD contributes probabilistic downside-event features to the risk envelope.

### Dataset loading

In [15]:
dataset = pd.read_csv("../../data/processed/financial_tools_datset.csv")

In [16]:
dataset["Price"]

0       1.1871
1       1.1895
2       1.1915
3       1.1827
4       1.1817
         ...  
1563    1.0834
1564    1.0830
1565    1.0840
1566    1.0871
1567    1.0914
Name: Price, Length: 1568, dtype: float64

### Return construction

Transform prices into log-returns so downside moves are measured on an additive and scale-consistent basis.

In [17]:
returns = np.log(dataset["Price"] / dataset["Price"].shift(1))

## 2. Model Construction

PDFD (Probability Density Function of Drawdowns) defines rolling downside-event frequencies under fixed negative thresholds.

In [24]:
risk_window = 252

In [25]:
drawdowns = returns.copy()
drawdowns[drawdowns > 0] = 0

### Drawdown thresholds

Each threshold encodes a severity level (mild to severe). Computing probabilities across thresholds yields a compact stress profile for each timestamp.

In [26]:
drawdown_levels = [
    -0.005,
    -0.01,
    -0.02,
    -0.03
]

In [27]:
pdfd_results = {}

for level in drawdown_levels:

    prob = (
        drawdowns
        .rolling(risk_window)
        .apply(lambda x: np.mean(x <= level), raw=True)
    )

    pdfd_results[level] = prob

In [28]:
pdfd_results[-0.005]
pdfd_results[-0.01]
pdfd_results[-0.02]
pdfd_results[-0.03]

0       NaN
1       NaN
2       NaN
3       NaN
4       NaN
       ... 
1563    0.0
1564    0.0
1565    0.0
1566    0.0
1567    0.0
Name: Price, Length: 1568, dtype: float64

## 3. Sanity Checks

Inspect threshold-wise probabilities to confirm ordering and plausibility before feature export.

In [29]:
for t in [-0.005, -0.01, -0.02, -0.03]:
    print(f"\nThreshold {t}")
    print(pdfd_results[t].dropna().head())


Threshold -0.005
252    0.130952
253    0.130952
254    0.130952
255    0.126984
256    0.130952
Name: Price, dtype: float64

Threshold -0.01
252    0.039683
253    0.039683
254    0.039683
255    0.039683
256    0.039683
Name: Price, dtype: float64

Threshold -0.02
252    0.003968
253    0.003968
254    0.003968
255    0.003968
256    0.003968
Name: Price, dtype: float64

Threshold -0.03
252    0.0
253    0.0
254    0.0
255    0.0
256    0.0
Name: Price, dtype: float64


## 4. Generation of Risk Features

Assemble threshold probabilities into a standardized feature table. Each column is a downside-event probability used by the probabilistic risk envelope.

In [30]:
pdfd_dataset = pd.DataFrame({
    "pdfd_005": pdfd_results[-0.005],
    "pdfd_01": pdfd_results[-0.01],
    "pdfd_02": pdfd_results[-0.02],
    "pdfd_03": pdfd_results[-0.03]
})

## 5. Output Standardization

Persist the PDFD risk-feature dataset for downstream merge with RL decision variables and other envelope components.

In [1]:
pdfd_dataset.to_csv(
    "../../data/processed/pdfd_dataset.csv",
    index=False
)

NameError: name 'pdfd_dataset' is not defined